In [3]:
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys, re

In [16]:
adata = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/raw/single-cell/sc_concat.h5ad')
st_data = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/raw/spatial/st_concat.h5ad')

In [ ]:
sys.path.append('/home/jwinchel/Syndri/Heimdall/')
from Heimdall.utils import convert_to_ensembl_ids

st_data,_ = convert_to_ensembl_ids(st_data, '/compute/oven-0-13/jwinchel/')
adata,_ = convert_to_ensembl_ids(adata, '/compute/oven-0-13/jwinchel/')
gene_overlap = set(st_data.var.index).intersection(adata.var.index)

zero_genes = set([st_data.var_names[i] for i in np.argwhere(st_data.X.sum(axis=0)==0).squeeze()] +
                [adata.var_names[i] for i in np.argwhere(adata.X.sum(axis=0)==0).squeeze()])
gene_overlap = list(gene_overlap.difference(zero_genes))


adata = adata[:,gene_overlap]
st_data = st_data[:,gene_overlap]
adata,st_data

human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 955 out of 958 identifiers (99.7%)
human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 955 out of 958 identifiers (99.7%)


(View of AnnData object with n_obs × n_vars = 40362 × 956
     obs: 'GSM_ID', 'source_count_file', 'Library', 'Barcode', 'Patient', 'Tissue', 'Barcoding emulsion', 'Total counts', 'Percent counts from mitochondrial genes', 'Most likely LM22 cell type', 'Major cell type', 'Minor subset', 'used_in_NSCLC_all_cells', 'x_NSCLC_all_cells', 'y_NSCLC_all_cells', 'used_in_NSCLC_and_blood_immune', 'x_NSCLC_and_blood_immune', 'y_NSCLC_and_blood_immune', 'used_in_NSCLC_immune', 'x_NSCLC_immune', 'y_NSCLC_immune', 'used_in_NSCLC_non_immune', 'x_NSCLC_non_immune', 'y_NSCLC_non_immune', 'used_in_blood', 'x_blood', 'y_blood', '_has_cell_metadata', 'shared_panel_total_counts'
     var: 'gene_symbol', 'gene_ensembl'
     uns: 'source_count_files', 'source_metadata_file', 'source_spatial_gene_panel_file', 'gene_mapping:symbol_to_ensembl', 'gene_mapping:ensembl_to_symbol',
 View of AnnData object with n_obs × n_vars = 800559 × 956
     obs: 'tissue', 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local

In [57]:
st_data = st_data[st_data.obs['cell_type']!='']
st_data.obs['cell_type'].value_counts()

cell_type
tumor 9         134099
fibroblast       93413
neutrophil       67726
tumor 6          66805
macrophage       53195
tumor 5          52286
plasmablast      38575
endothelial      38326
T CD4 naive      29327
B-cell           27512
tumor 13         26757
epithelial       24284
tumor 12         22442
T CD4 memory     16850
mDC              16796
pDC              13284
mast             11264
T CD8 naive      10565
monocyte          8296
NK                7875
Treg              6054
T CD8 memory      5505
Name: count, dtype: int64

In [73]:
st_data.obs['Run_Tissue_name'].unique()

['Lung12', 'Lung13', 'Lung5_Rep1', 'Lung5_Rep2', 'Lung5_Rep3', 'Lung6', 'Lung9_Rep1', 'Lung9_Rep2']
Categories (8, object): ['Lung5_Rep1', 'Lung5_Rep2', 'Lung5_Rep3', 'Lung6', 'Lung9_Rep1', 'Lung9_Rep2', 'Lung12', 'Lung13']

In [68]:
# adata.obs['Major cell type'].value_counts()
adata.obs['Major cell type'] = adata.obs['Major cell type'].apply(lambda x: x[1:] if x.startswith('t') else x)
adata.obs['Major cell type'].value_counts()

Major cell type
T cells                12776
MoMacDC                 9372
B cells                 5218
Neutrophils             2728
Plasma cells            2197
Patient1-specific       1307
NK cells                1116
Patient5-specific        766
Mast cells               732
Fibroblasts              585
Patient7-specific        538
Endothelial cells        535
Patient3-specific        527
Type II cells            403
Smooth muscle cells      318
pDC                      311
Patient6-specific        239
Patient4-specific        203
Patient2-specific        114
RBC                      108
ND                        85
Ciliated cells            70
Club cells                68
Type I cells              46
Name: count, dtype: int64

In [74]:
sys.path.append('/home/jwinchel/Syndri/code/')
import importlib
import mapping
importlib.reload(mapping)
from mapping import integrate_single_st_file_with_harmony

slices = list(st_data.obs['Run_Tissue_name'].unique())

st_data_list = []
for sl in slices:
    # print(sl,st_data.obs['slice'].unique(),len(st_data[st_data.obs['slice']==str(sl)]))
    adata,sl_adata, _ = integrate_single_st_file_with_harmony(st_data[st_data.obs['Run_Tissue_name']==str(sl)], adata,tissue_filter=None,n_pcs=30,tissue_name=str(sl))
    st_data_list.append(sl_adata)



/home/jwinchel/Syndri/code/mapping.py:62: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  st_data.obs["batch"] = "st"
/home/jwinchel/Syndri/code/mapping.py:65: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = sc_data.concatenate(
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


['GSM_ID', 'source_count_file', 'Library', 'Barcode', 'Patient', 'Tissue', 'Barcoding emulsion', 'Total counts', 'Percent counts from mitochondrial genes', 'Most likely LM22 cell type', 'Major cell type', 'Minor subset', 'used_in_NSCLC_all_cells', 'x_NSCLC_all_cells', 'y_NSCLC_all_cells', 'used_in_NSCLC_and_blood_immune', 'x_NSCLC_and_blood_immune', 'y_NSCLC_and_blood_immune', 'used_in_NSCLC_immune', 'x_NSCLC_immune', 'y_NSCLC_immune', 'used_in_NSCLC_non_immune', 'x_NSCLC_non_immune', 'y_NSCLC_non_immune', 'used_in_blood', 'x_blood', 'y_blood', '_has_cell_metadata', 'shared_panel_total_counts', 'batch', 'tissue', 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', '_has_slice_metadata', 'sdimx', 'sdimy', 'study_cell_ID', 'Area_study', 'AspectRatio_study', '

2026-09-14 22:06:46,516 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-09-14 22:07:19,917 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-09-14 22:07:20,502 - harmonypy - INFO - Iteration 1 of 30
2026-09-14 22:08:56,760 - harmonypy - INFO - Iteration 2 of 30
2026-09-14 22:10:15,562 - harmonypy - INFO - Converged after 2 iterations
/home/jwinchel/Syndri/code/mapping.py:62: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  st_data.obs["batch"] = "st"
/home/jwinchel/Syndri/code/mapping.py:65: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = sc_data.concatenate(
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return disp

['GSM_ID', 'source_count_file', 'Library', 'Barcode', 'Patient', 'Tissue', 'Barcoding emulsion', 'Total counts', 'Percent counts from mitochondrial genes', 'Most likely LM22 cell type', 'Major cell type', 'Minor subset', 'used_in_NSCLC_all_cells', 'x_NSCLC_all_cells', 'y_NSCLC_all_cells', 'used_in_NSCLC_and_blood_immune', 'x_NSCLC_and_blood_immune', 'y_NSCLC_and_blood_immune', 'used_in_NSCLC_immune', 'x_NSCLC_immune', 'y_NSCLC_immune', 'used_in_NSCLC_non_immune', 'x_NSCLC_non_immune', 'y_NSCLC_non_immune', 'used_in_blood', 'x_blood', 'y_blood', '_has_cell_metadata', 'shared_panel_total_counts', 'batch', 'tissue', 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', '_has_slice_metadata', 'sdimx', 'sdimy', 'study_cell_ID', 'Area_study', 'AspectRatio_study', '

2026-09-14 22:11:06,188 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-09-14 22:11:06,803 - harmonypy - INFO - Iteration 1 of 30
2026-09-14 22:12:26,393 - harmonypy - INFO - Iteration 2 of 30
2026-09-14 22:13:22,057 - harmonypy - INFO - Iteration 3 of 30
2026-09-14 22:14:53,561 - harmonypy - INFO - Iteration 4 of 30
2026-09-14 22:15:36,489 - harmonypy - INFO - Iteration 5 of 30
2026-09-14 22:16:29,539 - harmonypy - INFO - Iteration 6 of 30
2026-09-14 22:18:55,299 - harmonypy - INFO - Iteration 7 of 30
2026-09-14 22:20:16,735 - harmonypy - INFO - Iteration 8 of 30
2026-09-14 22:22:29,587 - harmonypy - INFO - Iteration 9 of 30
2026-09-14 22:24:03,318 - harmonypy - INFO - Iteration 10 of 30
2026-09-14 22:24:38,235 - harmonypy - INFO - Iteration 11 of 30
2026-09-14 22:25:19,661 - harmonypy - INFO - Converged after 11 iterations
/home/jwinchel/Syndri/code/mapping.py:62: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  

['GSM_ID', 'source_count_file', 'Library', 'Barcode', 'Patient', 'Tissue', 'Barcoding emulsion', 'Total counts', 'Percent counts from mitochondrial genes', 'Most likely LM22 cell type', 'Major cell type', 'Minor subset', 'used_in_NSCLC_all_cells', 'x_NSCLC_all_cells', 'y_NSCLC_all_cells', 'used_in_NSCLC_and_blood_immune', 'x_NSCLC_and_blood_immune', 'y_NSCLC_and_blood_immune', 'used_in_NSCLC_immune', 'x_NSCLC_immune', 'y_NSCLC_immune', 'used_in_NSCLC_non_immune', 'x_NSCLC_non_immune', 'y_NSCLC_non_immune', 'used_in_blood', 'x_blood', 'y_blood', '_has_cell_metadata', 'shared_panel_total_counts', 'batch', 'tissue', 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', '_has_slice_metadata', 'sdimx', 'sdimy', 'study_cell_ID', 'Area_study', 'AspectRatio_study', '

... (7 outputs left out)

['GSM_ID', 'source_count_file', 'Library', 'Barcode', 'Patient', 'Tissue', 'Barcoding emulsion', 'Total counts', 'Percent counts from mitochondrial genes', 'Most likely LM22 cell type', 'Major cell type', 'Minor subset', 'used_in_NSCLC_all_cells', 'x_NSCLC_all_cells', 'y_NSCLC_all_cells', 'used_in_NSCLC_and_blood_immune', 'x_NSCLC_and_blood_immune', 'y_NSCLC_and_blood_immune', 'used_in_NSCLC_immune', 'x_NSCLC_immune', 'y_NSCLC_immune', 'used_in_NSCLC_non_immune', 'x_NSCLC_non_immune', 'y_NSCLC_non_immune', 'used_in_blood', 'x_blood', 'y_blood', '_has_cell_metadata', 'shared_panel_total_counts', 'batch', 'tissue', 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', '_has_slice_metadata', 'sdimx', 'sdimy', 'study_cell_ID', 'Area_study', 'AspectRatio_study', '

2026-09-14 23:04:47,963 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-09-14 23:04:48,620 - harmonypy - INFO - Iteration 1 of 30
2026-09-14 23:05:57,854 - harmonypy - INFO - Iteration 2 of 30
2026-09-14 23:07:58,361 - harmonypy - INFO - Iteration 3 of 30
2026-09-14 23:09:25,149 - harmonypy - INFO - Iteration 4 of 30
2026-09-14 23:10:28,337 - harmonypy - INFO - Iteration 5 of 30
2026-09-14 23:11:50,680 - harmonypy - INFO - Iteration 6 of 30
2026-09-14 23:13:20,570 - harmonypy - INFO - Iteration 7 of 30
2026-09-14 23:14:54,018 - harmonypy - INFO - Iteration 8 of 30
2026-09-14 23:16:32,312 - harmonypy - INFO - Iteration 9 of 30
2026-09-14 23:17:05,370 - harmonypy - INFO - Iteration 10 of 30
2026-09-14 23:17:56,239 - harmonypy - INFO - Converged after 10 iterations
/home/jwinchel/Syndri/code/mapping.py:62: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  st_data.obs["batch"] = "st"
/home/jwinchel/Syndri/code/mapping.p

['GSM_ID', 'source_count_file', 'Library', 'Barcode', 'Patient', 'Tissue', 'Barcoding emulsion', 'Total counts', 'Percent counts from mitochondrial genes', 'Most likely LM22 cell type', 'Major cell type', 'Minor subset', 'used_in_NSCLC_all_cells', 'x_NSCLC_all_cells', 'y_NSCLC_all_cells', 'used_in_NSCLC_and_blood_immune', 'x_NSCLC_and_blood_immune', 'y_NSCLC_and_blood_immune', 'used_in_NSCLC_immune', 'x_NSCLC_immune', 'y_NSCLC_immune', 'used_in_NSCLC_non_immune', 'x_NSCLC_non_immune', 'y_NSCLC_non_immune', 'used_in_blood', 'x_blood', 'y_blood', '_has_cell_metadata', 'shared_panel_total_counts', 'batch', 'tissue', 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', '_has_slice_metadata', 'sdimx', 'sdimy', 'study_cell_ID', 'Area_study', 'AspectRatio_study', '

2026-09-14 23:18:57,467 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-09-14 23:18:58,578 - harmonypy - INFO - Iteration 1 of 30
2026-09-14 23:20:44,590 - harmonypy - INFO - Iteration 2 of 30
2026-09-14 23:21:57,933 - harmonypy - INFO - Iteration 3 of 30
2026-09-14 23:23:14,377 - harmonypy - INFO - Iteration 4 of 30
2026-09-14 23:24:29,667 - harmonypy - INFO - Iteration 5 of 30
2026-09-14 23:25:54,848 - harmonypy - INFO - Iteration 6 of 30
2026-09-14 23:27:09,935 - harmonypy - INFO - Iteration 7 of 30
2026-09-14 23:28:30,438 - harmonypy - INFO - Iteration 8 of 30
2026-09-14 23:29:45,420 - harmonypy - INFO - Iteration 9 of 30
2026-09-14 23:30:58,461 - harmonypy - INFO - Iteration 10 of 30
2026-09-14 23:31:44,417 - harmonypy - INFO - Iteration 11 of 30
2026-09-14 23:32:09,783 - harmonypy - INFO - Iteration 12 of 30
2026-09-14 23:32:41,867 - harmonypy - INFO - Iteration 13 of 30
2026-09-14 23:33:05,426 - harmonypy - INFO - Converged after 13 iterations


In [76]:
adata.write_h5ad('/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/interim/single-cell/sc_integrated.h5ad')

In [75]:
for sl in slices:
    st_data[st_data.obs['Run_Tissue_name']==sl].write_h5ad(f'/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/interim/spatial/st_integrated_slice-{sl}.h5ad')

/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.va

In [89]:
adata.obs['cell_type'] = adata.obs['Major cell type'].copy()
adata.obs['cell_type'] = adata.obs['cell_type'].str.replace(r"^Patient\d+.*", "tumor", regex=True)
sorted(adata.obs['cell_type'].unique())

['B cells',
 'Ciliated cells',
 'Club cells',
 'Endothelial cells',
 'Fibroblasts',
 'Mast cells',
 'MoMacDC',
 'ND',
 'NK cells',
 'Neutrophils',
 'Plasma cells',
 'RBC',
 'Smooth muscle cells',
 'T cells',
 'Type I cells',
 'Type II cells',
 'pDC',
 'tumor']

In [87]:
st_data.obs['cell_type_orig'] = st_data.obs['cell_type'].copy()
sorted(st_data.obs['cell_type_orig'].unique()),sorted(adata.obs['cell_type'].unique())

(['B-cell',
  'NK',
  'T CD4 memory',
  'T CD4 naive',
  'T CD8 memory',
  'T CD8 naive',
  'Treg',
  'endothelial',
  'epithelial',
  'fibroblast',
  'mDC',
  'macrophage',
  'mast',
  'monocyte',
  'neutrophil',
  'pDC',
  'plasmablast',
  'tumor 12',
  'tumor 13',
  'tumor 5',
  'tumor 6',
  'tumor 9'],
 ['B cells',
  'Ciliated cells',
  'Club cells',
  'Endothelial cells',
  'Fibroblasts',
  'Mast cells',
  'MoMacDC',
  'ND',
  'NK cells',
  'Neutrophils',
  'Patient1-specific',
  'Patient2-specific',
  'Patient3-specific',
  'Patient4-specific',
  'Patient5-specific',
  'Patient6-specific',
  'Patient7-specific',
  'Plasma cells',
  'RBC',
  'Smooth muscle cells',
  'T cells',
  'Type I cells',
  'Type II cells',
  'pDC'])

In [92]:
TARGET_TYPES = [
    "Malignant",
    "Fibroblast",
    "Macrophage",
    "Monocyte",
    "DC",
    "Mast",
    "Treg",
    "T CD4+",
    "T CD8+",
    "NK",
    "B cells",
    "Plasmablast",
    "Neutrophil",
    "Endothelial",
    "Epithelial",
]

SC_DIRECT_MAP = {
    "B cells": "B cells",
    "Ciliated cells": "Epithelial",
    "Club cells": "Epithelial",
    "Endothelial cells": "Endothelial",
    "Fibroblasts": "Fibroblast",
    "Mast cells": "Mast",
    "NK cells": "NK",
    "Neutrophils": "Neutrophil",
    "Plasma cells": "Plasmablast",
    "Type I cells": "Epithelial",
    "Type II cells": "Epithelial",
    "pDC": "DC",

    # Assuming these are malignant populations.
    "Patient1-specific": "Malignant",
    "Patient2-specific": "Malignant",
    "Patient3-specific": "Malignant",
    "Patient4-specific": "Malignant",
    "Patient5-specific": "Malignant",
    "Patient6-specific": "Malignant",
    "Patient7-specific": "Malignant",
}

ST_MAP = {
    "B-cell": "B cells",
    "NK": "NK",

    "T CD4 memory": "T CD4+",
    "T CD4 naive": "T CD4+",
    "T CD8 memory": "T CD8+",
    "T CD8 naive": "T CD8+",
    "Treg": "Treg",

    "endothelial": "Endothelial",
    "epithelial": "Epithelial",
    "fibroblast": "Fibroblast",

    "mDC": "DC",
    "pDC": "DC",
    "macrophage": "Macrophage",
    "monocyte": "Monocyte",

    "mast": "Mast",
    "neutrophil": "Neutrophil",
    "plasmablast": "Plasmablast",

    "tumor 5": "Malignant",
    "tumor 6": "Malignant",
    "tumor 9": "Malignant",
    "tumor 12": "Malignant",
    "tumor 13": "Malignant",
}

In [145]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
def harmonize_reference_for_slice(
    st,
    sc,
    st_harmony_key,
    sc_harmony_key,
    st_label_col="cell_type",
    sc_label_col="cell_type",
    k=3,
):
    # Start with the labels that can be mapped deterministically
    labels = sc.obs[sc_label_col].map(SC_DIRECT_MAP).astype(object)

    # Harmonized spatial labels
    st_labels = st.obs[st_label_col].map(ST_MAP)

    # --------------------
    # T cells
    # --------------------
    sc_mask = sc.obs[sc_label_col].eq("T cells")
    st_mask = st_labels.isin(["T CD4+", "T CD8+", "Treg"])

    if sc_mask.any() and st_mask.any():
        clf = KNeighborsClassifier(
            n_neighbors=min(k, st_mask.sum()),
            weights="distance",
            metric="euclidean",
        )

        clf.fit(
            st.obsm[st_harmony_key][st_mask],
            st_labels.loc[st_mask],
        )

        labels.loc[sc_mask] = clf.predict(
            sc.obsm[sc_harmony_key][sc_mask]
        )

    # --------------------
    # MoMacDC
    # --------------------
    sc_mask = sc.obs[sc_label_col].eq("MoMacDC")
    st_mask = st_labels.isin(["Macrophage", "Monocyte", "DC"])

    if sc_mask.any() and st_mask.any():
        clf = KNeighborsClassifier(
            n_neighbors=min(k, st_mask.sum()),
            weights="distance",
            metric="euclidean",
        )

        clf.fit(
            st.obsm[st_harmony_key][st_mask],
            st_labels.loc[st_mask],
        )

        labels.loc[sc_mask] = clf.predict(
            sc.obsm[sc_harmony_key][sc_mask]
        )

    return labels

In [150]:
for i, st in enumerate(st_data_list):
    sc_key = list(st.obsm.keys())[0]
    # sc_key = f"X_pca_harmony_{i}"

    adata.obs[f"cell_type_harmonized_{i}"] = (
        harmonize_reference_for_slice(
            st=st,
            sc=adata,
            st_harmony_key=sc_key,
            sc_harmony_key=sc_key,
            st_label_col="cell_type",
            sc_label_col="Major cell type",
            k=1,
        )
    )

    # Also give the spatial object its harmonized ontology
    st.obs["cell_type_harmonized"] = (
        st.obs["cell_type"].map(ST_MAP)
    )

In [151]:
harm_cols = [
    c for c in adata.obs.columns
    if c.startswith("cell_type_harmonized_")
]

adata.obs["harmonized_consensus"] = (
    adata.obs[harm_cols]
    .mode(axis=1)[0]
)

adata.obs["harmonized_agreement"] = (
    adata.obs[harm_cols]
    .eq(adata.obs["harmonized_consensus"], axis=0)
    .mean(axis=1)
)

In [153]:
adata.obs.harmonized_consensus.value_counts()

harmonized_consensus
T CD4+         9374
Macrophage     5714
B cells        5218
Malignant      3694
T CD8+         3383
DC             2942
Neutrophil     2728
Plasmablast    2197
NK             1116
Monocyte       1027
Mast            732
Epithelial      587
Fibroblast      585
Endothelial     535
Treg             19
Name: count, dtype: int64

In [161]:
adata.obs['cell_type_harmonized'] = adata.obs['harmonized_consensus']

In [162]:
adata.write_h5ad('/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/interim/single-cell/sc_integrated.h5ad')

In [163]:
for sl in slices:
    st_data[st_data.obs['Run_Tissue_name']==sl].write_h5ad(f'/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/interim/spatial/st_integrated_slice-{sl}.h5ad')

/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.va

In [164]:
adata.obsm

AxisArrays with keys: X_pca_harmony_Lung12, X_pca_harmony_Lung13, X_pca_harmony_Lung5_Rep1, X_pca_harmony_Lung5_Rep2, X_pca_harmony_Lung5_Rep3, X_pca_harmony_Lung6, X_pca_harmony_Lung9_Rep1, X_pca_harmony_Lung9_Rep2

In [182]:
st_data = st_data_list[-2].copy()
st_data

AnnData object with n_obs × n_vars = 87606 × 956
    obs: 'tissue', 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', '_has_slice_metadata', 'sdimx', 'sdimy', 'study_cell_ID', 'Area_study', 'AspectRatio_study', 'Width_study', 'Height_study', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI_study', 'Max.DAPI_study', 'dualfiles', 'Slide_name', 'study_tissue', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors', 'batch', 'cell_type_harmonized'
    var: 'gene_symbol', 'gene_ensembl'
    uns: 'source_exprMat_files', 'source_slice_me

In [183]:
st_data.obsm['X_pca_harmony'] = st_data.obsm['X_pca_harmony_Lung9_Rep1']
adata.obsm['X_pca_harmony'] = adata.obsm['X_pca_harmony_Lung9_Rep1']


In [184]:
st_data

AnnData object with n_obs × n_vars = 87606 × 956
    obs: 'tissue', 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', '_has_slice_metadata', 'sdimx', 'sdimy', 'study_cell_ID', 'Area_study', 'AspectRatio_study', 'Width_study', 'Height_study', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI_study', 'Max.DAPI_study', 'dualfiles', 'Slide_name', 'study_tissue', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors', 'batch', 'cell_type_harmonized'
    var: 'gene_symbol', 'gene_ensembl'
    uns: 'source_exprMat_files', 'source_slice_me

In [185]:
import sys
import numpy as np
sys.path.append('/home/jwinchel/Syndri/code')
from mapping import map_sc_reference_to_unique_xenium_cells

sc_mapped = map_sc_reference_to_unique_xenium_cells(
    # adata_subset,
    adata,
    st_data,
    embedding_key="X_pca_harmony",
    spatial_cols=('CenterX_global_px', 'CenterY_global_px'),
    n_matches=50,
    reuse_penalty=9.0,
    rng=np.random.default_rng(0),
    max_reuse_per_sc=5000,
    match_cell_type=True,
    ensure_all_sc_used=True,
    sc_cell_type_col="cell_type_harmonized",
    st_cell_type_col="cell_type_harmonized",
)

/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/jwinchel/Syndri/code/mapping.py:636: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mapped_sc_data.obs[f"{prefix}_{j+1}_id"] = match_result["ids"][:, j]
/home/jwinchel/Syndri/code/mapping.py:637: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mapped_sc_data.obs[f"{prefix}_{j+1}_dist"] = match_resu

In [186]:
sc_mapped

AnnData object with n_obs × n_vars = 87606 × 956
    obs: 'GSM_ID', 'source_count_file', 'Library', 'Barcode', 'Patient', 'Tissue', 'Barcoding emulsion', 'Total counts', 'Percent counts from mitochondrial genes', 'Most likely LM22 cell type', 'Major cell type', 'Minor subset', 'used_in_NSCLC_all_cells', 'x_NSCLC_all_cells', 'y_NSCLC_all_cells', 'used_in_NSCLC_and_blood_immune', 'x_NSCLC_and_blood_immune', 'y_NSCLC_and_blood_immune', 'used_in_NSCLC_immune', 'x_NSCLC_immune', 'y_NSCLC_immune', 'used_in_NSCLC_non_immune', 'x_NSCLC_non_immune', 'y_NSCLC_non_immune', 'used_in_blood', 'x_blood', 'y_blood', '_has_cell_metadata', 'shared_panel_total_counts', 'batch', 'cell_type', 'cell_type_harmonized_0', 'cell_type_harmonized_1', 'cell_type_harmonized_2', 'cell_type_harmonized_3', 'cell_type_harmonized_4', 'cell_type_harmonized_5', 'cell_type_harmonized_6', 'cell_type_harmonized_7', 'harmonized_consensus', 'harmonized_agreement', 'cell_type_harmonized', 'source_sc_id', 'sampled_sc_id', 'match

In [187]:
sc_mapped.uns['sc_reference_to_spatial_mapping']['spatial_cols'] = list(sc_mapped.uns['sc_reference_to_spatial_mapping']['spatial_cols'])
sc_mapped.obs['split'] = 'train'
splits = adata.obs.groupby('cell_type_harmonized').cumcount().apply(lambda x: x % 5)
val_ids = splits[splits==4].index.tolist()


sc_mapped.obs.loc[sc_mapped.obs['source_sc_id'].isin(val_ids),'split'] = 'val'

# cell type
cell_types = sorted(sc_mapped.obs['cell_type_harmonized'].unique())
cell_type_dict = {cell_types[i]: -(i+1) for i in range(len(cell_types))}
sc_mapped.uns['cell_type_dict'] = cell_type_dict
# sc_mapped.obs['cell_type'] = sc_mapped.obs['']
sc_mapped.write_h5ad('/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/interim/single-cell/sc_integrated_mapped-Lung9-Rep1.h5ad')

/tmp/ipykernel_1450859/4090009196.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sc_mapped.obs['split'] = 'train'
/tmp/ipykernel_1450859/4090009196.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  splits = adata.obs.groupby('cell_type_harmonized').cumcount().apply(lambda x: x % 5)


In [188]:
sc_mapped.obs['cell_type_harmonized'].value_counts()

cell_type_harmonized
Malignant      39422
Fibroblast     13952
Neutrophil      5948
Endothelial     4428
Macrophage      4080
T CD4+          4036
Epithelial      3584
Plasmablast     2895
T CD8+          2323
DC              2059
Monocyte        1702
Treg            1269
B cells          871
NK               746
Mast             291
Name: count, dtype: int64